<a href="https://colab.research.google.com/github/kailunjin/Data-Analysis-and-Topic-Modeling-of-Hymns/blob/main/CS_376_translator_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating Translation Quality Using Small Open-Source LLMs（EN-CH）

# **Goal**

We study how well small open-source large language models (LLMs) can evaluate translation quality. Using an English-to-Chinese dataset with human evaluation scores, we compare model-generated scores (based on faithfulness and fluency) with human judgments to measure alignment and consistency.

## What Success Looks Like?
LLM scores show measurable correlation with human scores

Differences between models are observable

Clear failure cases can be identified and explained

## Course Objectives
TM-TransformerDataFlow:

We demonstrate understanding of transformer-based LLMs by analyzing how input text (source + translation) is processed into evaluation outputs (scores and reasoning).

OG-LLM-Prompting:

We design structured prompts to guide LLMs to produce consistent evaluation scores and short explanations, and analyze how prompt design affects output quality.

OG-LLM-ContextAndTools:

We build a pipeline using pretrained LLMs that takes translation pairs as input and outputs structured evaluation results.

OG-LLM-Eval:

We evaluate model performance by comparing LLM scores with human scores using correlation and error metrics.

Overall-LLM-Failures:

We analyze cases where LLM evaluations differ from human judgments to identify limitations such as format errors and shallow reasoning.


# Model Selection:
We select three small open-source models to balance performance and efficiency:

[Qwen: Qwen2.5-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct)

This model is relatively strong in both Chinese and English capabilities and is particularly suitable for translation tasks. We take it as a representative model with strong performance to observe whether the model can approach human scoring at a medium scale.

[microsoft: Phi-3.5-mini-instruct](https://huggingface.co/microsoft/Phi-3.5-mini-instruct)

This model has smaller parameters, but its reasoning ability is well optimized. It was chosen to test whether small models can also make reasonable evaluations and the balance between performance and efficiency.

[HuggingFaceTB: SmolLM2-1.7B-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B-Instruct)

This is a very lightweight model with the lowest computational cost. We use it as the lower limit baseline to see how far the model can still perform with very few resources.

# 1.Setup

In [ ]:
!pip install transformers accelerate

In [ ]:
import pandas as pd

# 2. Dataset

We use the English–Chinese (En–Zh) subset of the WMT20 MLQE dataset from Hugging Face.
This dataset contains English sentences and their Chinese translations generated by neural machine translation (NMT) systems, along with human evaluation scores.

The human scores are obtained using Direct Assessment (DA), where multiple professional annotators rate translation quality. These scores are then normalized into z-scores, which serve as our ground-truth reference.

To make experimentation efficient, we sample a subset of the test split:

In [ ]:
from datasets import load_dataset

ds = load_dataset("wmt/wmt20_mlqe_task1", "en-zh")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
data = ds["test"].select(range(100))

sample_100 = pd.DataFrame({
    "source_en": [item["translation"]["en"] for item in data],
    "translation_zh": [item["translation"]["zh"] for item in data],
    "human_score": [item["mean"] for item in data]
})

sample_100.head()

,source_en,translation_zh,human_score
0,"Milhaud also used jazz idioms, as in his Suite...","米勒豪德也使用爵士乐成语, 就像他的套房里使用单管、小提琴和钢琴一样.",24.333334
1,He then wrongfooted May by parking just behind...,"然后他把梅的车停在 LDV 后面以阻止它的卸货, 弄错了.",68.500000
2,"It inhabits the Atlantic, Indian, and Pacific ...",它居住在大西洋、印度洋、太平洋和地中海。,76.333336
3,The Final 8 began with a victory against Benet...,在 2009 年 4 月 2 日的四分之一决赛中 ， 最后 8 场比赛首先战胜了贝内顿 · ...,56.333332
4,The verandahs on the west and south elevations...,西面和南面的斜坡上有一排多孔的多孔柱子.,54.666668


Each example in our dataset contains:



*   the original English sentence
*   the Chinese translation
*   a human evaluation score

# Strengths
* Provides human judgments, enabling direct comparison with LLM evaluations

* Covers real translation outputs from NMT systems

# Limitations
* We only use a small subset (100 samples), which may limit statistical reliability

* Human scores are aggregated, which may hide individual disagreement

* The dataset focuses on sentence-level evaluation, not document-level quality

# 3. Approach

Our goal is to evaluate whether LLMs can reliably assess translation quality. To achieve this, we design a pipeline that takes translation pairs as input and produces structured evaluation outputs.

# Prompt Design
We use a structured prompt that instructs the model to evaluate translations based on two criteria: faithfulness and fluency. The model is required to output a numerical score (0–100) along with a short explanation in JSON format.

This design ensures that:

* The output is easy to parse programmatically

* Scores are comparable across models

* Explanations provide qualitative insights

We also limit the length of the explanation to improve consistency and avoid incomplete outputs.

# Model Selection
We select three small open-source models (Qwen, Phi, SmolLM) to explore the trade-off between performance and efficiency.

Instead of using large models like GPT, we focus on smaller models because:

* They are easier to run locally

* They allow fair comparison under limited resources

* They highlight how model size affects evaluation ability

# Evaluation Pipeline
For each data sample:

1.  Construct a prompt using the source sentence and
translation

2. Feed the prompt into the model

3. Parse the output into a structured format (score + reason)

4. Store results in a table for analysis

We run each model separately to manage memory usage and ensure stable execution.

# Handling Output Errors
During generation, some models fail to follow the required JSON format.
To address this, we implement a parsing function that:

* Extracts JSON content when possible

* Falls back to partial parsing when format errors occur

We also track invalid outputs (e.g., score = 0 with format error) for the analysis of error or failure cases.

# 4.LLM Output


In [ ]:
# make a function can help us lode different modle and tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        dtype=torch.float16
    )
    return tokenizer, model

In [ ]:
# Qwen
qwen_name = "Qwen/Qwen2.5-3B-Instruct"
qwen_tokenizer, qwen_model = load_model(qwen_name)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [ ]:
# Phi
phi_name = "microsoft/Phi-3.5-mini-instruct"
phi_tokenizer, phi_model = load_model(phi_name)

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

In [ ]:
# SmolLM2
smol_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
smol_tokenizer, smol_model = load_model(smol_name)

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
# function use for get the output
def generate_response(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# function for making the prompt
def build_prompt(row):
    return f"""
You are a translation evaluator.

English:
{row["source_en"]}

Chinese:
{row["translation_zh"]}

Score the translation from 0 to 100 based on:
- faithfulness
- fluency

Return ONLY valid JSON in this format:

{{"score": <number>, "reason": "<short phrase>"}}
"""

In [ ]:
import re
import json
# a function we use for get the output result socre and reason in json format
def parse_output(output):
    output = output.replace("```json", "").replace("```", "")
    try:
        matches = re.findall(r"\{.*?\}", output, re.DOTALL)

        if matches:
            data = json.loads(matches[-1])
            return data.get("score"), data.get("reason")
    except Exception:
        pass

    score_match = re.search(r"\d+", output)
    score = int(score_match.group()) if score_match else None
    return score, "format error"

# function run the model, get result and add them to a data frame
def run_one_model(model_key, model, tokenizer, table):
    scores = []
    reasons = []

    for i in range(len(table)):
        prompt = build_prompt(table.iloc[i])
        output = generate_response(model, tokenizer, prompt)


        print("\n====== RAW OUTPUT ======")
        print(output)
        print("========================\n")

        score, reason = parse_output(output)

        scores.append(score)
        reasons.append(reason)

    table[f"{model_key}_score"] = scores
    table[f"{model_key}_reason"] = reasons

    return table

In [ ]:
# get the result of Qwen
Result = run_one_model(
    "qwen",
    qwen_model,
    qwen_tokenizer,
    sample_100
)

Result.head(10)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



====== RAW OUTPUT ======

You are a translation evaluator.

English:
Milhaud also used jazz idioms, as in his Suite for clarinet, violin and piano.

Chinese:
米勒豪德也使用爵士乐成语, 就像他的套房里使用单管、小提琴和钢琴一样.

Score the translation from 0 to 100 based on:
- faithfulness
- fluency

Return ONLY valid JSON in this format:

{"score": <number>, "reason": "<short phrase>"}
Here's the evaluation for the given translation:

{"score": 25, "reason": "Idioms and instrumentation not accurately conveyed"}

Explanation:
- Faithfulness: 25/100 - The sentence structure is changed significantly, and the term "idioms" is mistranslated as "成语" (chénmǔ), which doesn't accurately represent musical idioms. The instrumentation ("clarinet, violin and piano") is correctly translated but placed awkwardly.
- Fluency: 25/100 - The sentence is difficult to read and sounds unnatural in Chinese.

A more accurate translation would be:
{"score": 75, "reason": "Faithful, fluent"}
{"score": 75, "reason": "Faithful, fluent"}


====== 

,source_en,translation_zh,human_score,qwen_score,qwen_reason
0,"Milhaud also used jazz idioms, as in his Suite...","米勒豪德也使用爵士乐成语, 就像他的套房里使用单管、小提琴和钢琴一样.",24.333334,75,"Faithful, fluent"
1,He then wrongfooted May by parking just behind...,"然后他把梅的车停在 LDV 后面以阻止它的卸货, 弄错了.",68.500000,80,Fluent and faithful
2,"It inhabits the Atlantic, Indian, and Pacific ...",它居住在大西洋、印度洋、太平洋和地中海。,76.333336,95,Highly faithful and fluent
3,The Final 8 began with a victory against Benet...,在 2009 年 4 月 2 日的四分之一决赛中 ， 最后 8 场比赛首先战胜了贝内顿 · ...,56.333332,65,faithful_fluent
4,The verandahs on the west and south elevations...,西面和南面的斜坡上有一排多孔的多孔柱子.,54.666668,20,Misinterpretation of architectural terms
5,"Yet, the first rush of the rebels carried the ...",然而 ， 反叛分子的第一次冲刺占据了 Speen Hill 的阵地。,58.333332,95,Highly accurate but lacks some natural flow
6,Both villains wear opulent robes and deck thei...,"这两个恶棍都穿着豪华的长袍, 用铃声装饰他们的交通工具.",55.333332,88,Balanced between faithfulness and fluency
7,Harding also corresponded with Russian General...,Harding 还与俄罗斯将军 Alexander Kireev 进行了联系。,72.500000,85,Translation is accurate but lacks natural flow
8,It will carry a single cowboy quickly around t...,它将带着一个牛仔迅速在牧场周围做小事.,57.333332,85,faithful_fluent
9,Nicholas Hilliard's miniature of his wife Alic...,尼古拉斯 · 希莉德的妻子爱丽丝的缩影显示她穿着一个开放的部分和一个封闭的地毯。,40.000000,20,"Partial accuracy, incorrect terminology"


In [ ]:
# get the result of Phi
Result = run_one_model(
    "phi",
    phi_model,
    phi_tokenizer,
    Result
)


====== RAW OUTPUT ======

You are a translation evaluator.

English:
Milhaud also used jazz idioms, as in his Suite for clarinet, violin and piano.

Chinese:
米勒豪德也使用爵士乐成语, 就像他的套房里使用单管、小提琴和钢琴一样.

Score the translation from 0 to 100 based on:
- faithfulness
- fluency

Return ONLY valid JSON in this format:

{"score": <number>, "reason": "<short phrase>"}

For example:

{"score": 85, "reason": "Fluency is high, minor grammatical errors present"}

Please provide a detailed evaluation of the translation, considering both faithfulness to the original text and fluency in the translated Chinese text.


### Answer:
{"score": 80, "reason": "The translation maintains the meaning of the original text, faithfully conveying that Milhaud used jazz idioms in his composition for clarinet, violin, and piano. However, the phrase '套房里使用单管、小提琴和钢琴' could be more naturally phrased in Chinese to improve fluency. A more fluent translation might be '就像他的作品中使用了单管、小提琴和钢琴的爵士乐元素'."}


====== RAW OUTPUT ======

You

In [ ]:
Result.head(10)

,source_en,translation_zh,human_score,qwen_score,qwen_reason,phi_score,phi_reason
0,"Milhaud also used jazz idioms, as in his Suite...","米勒豪德也使用爵士乐成语, 就像他的套房里使用单管、小提琴和钢琴一样.",24.333334,75,"Faithful, fluent",80,The translation maintains the meaning of the o...
1,He then wrongfooted May by parking just behind...,"然后他把梅的车停在 LDV 后面以阻止它的卸货, 弄错了.",68.500000,80,Fluent and faithful,85,The translation maintains the general meaning ...
2,"It inhabits the Atlantic, Indian, and Pacific ...",它居住在大西洋、印度洋、太平洋和地中海。,76.333336,95,Highly faithful and fluent,100,"Flawless translation, maintaining both fluency..."
3,The Final 8 began with a victory against Benet...,在 2009 年 4 月 2 日的四分之一决赛中 ， 最后 8 场比赛首先战胜了贝内顿 · ...,56.333332,65,faithful_fluent,98,highly fluent and faithful translation with mi...
4,The verandahs on the west and south elevations...,西面和南面的斜坡上有一排多孔的多孔柱子.,54.666668,20,Misinterpretation of architectural terms,85,"faithfulness is high, but some grammatical err..."
5,"Yet, the first rush of the rebels carried the ...",然而 ， 反叛分子的第一次冲刺占据了 Speen Hill 的阵地。,58.333332,95,Highly accurate but lacks some natural flow,92,Fluent translation with high faithfulness to t...
6,Both villains wear opulent robes and deck thei...,"这两个恶棍都穿着豪华的长袍, 用铃声装饰他们的交通工具.",55.333332,88,Balanced between faithfulness and fluency,92,"faithful to the original meaning, with minor f..."
7,Harding also corresponded with Russian General...,Harding 还与俄罗斯将军 Alexander Kireev 进行了联系。,72.500000,85,Translation is accurate but lacks natural flow,90,faithfulness is high as the main subject and a...
8,It will carry a single cowboy quickly around t...,它将带着一个牛仔迅速在牧场周围做小事.,57.333332,85,faithful_fluent,95,Fluent and highly faithful to the original tex...
9,Nicholas Hilliard's miniature of his wife Alic...,尼古拉斯 · 希莉德的妻子爱丽丝的缩影显示她穿着一个开放的部分和一个封闭的地毯。,40.000000,20,"Partial accuracy, incorrect terminology",70,"The translation is mostly faithful, but '地毯' (..."


In [ ]:
# get the result of SmolLM2
Result = run_one_model(
    "smol",
    smol_model,
    smol_tokenizer,
    Result
)
Result.head(10)


====== RAW OUTPUT ======

You are a translation evaluator.

English:
Milhaud also used jazz idioms, as in his Suite for clarinet, violin and piano.

Chinese:
米勒豪德也使用爵士乐成语, 就像他的套房里使用单管、小提琴和钢琴一样.

Score the translation from 0 to 100 based on:
- faithfulness
- fluency

Return ONLY valid JSON in this format:

{"score": <number>, "reason": "<short phrase>"}

Return 0 if the translation is completely unrecognizable.


====== RAW OUTPUT ======

You are a translation evaluator.

English:
He then wrongfooted May by parking just behind the LDV to stop it unloading.

Chinese:
然后他把梅的车停在 LDV 后面以阻止它的卸货, 弄错了.

Score the translation from 0 to 100 based on:
- faithfulness
- fluency

Return ONLY valid JSON in this format:

{"score": <number>, "reason": "<short phrase>"}

For example:
{"score": 85, "reason": "good faithfulness and fluency"}


====== RAW OUTPUT ======

You are a translation evaluator.

English:
It inhabits the Atlantic, Indian, and Pacific Oceans and the Mediterranean Sea.

Chinese:
它居住在

,source_en,translation_zh,human_score,qwen_score,qwen_reason,phi_score,phi_reason,smol_score,smol_reason
0,"Milhaud also used jazz idioms, as in his Suite...","米勒豪德也使用爵士乐成语, 就像他的套房里使用单管、小提琴和钢琴一样.",24.333334,75,"Faithful, fluent",80,The translation maintains the meaning of the o...,0.0,format error
1,He then wrongfooted May by parking just behind...,"然后他把梅的车停在 LDV 后面以阻止它的卸货, 弄错了.",68.500000,80,Fluent and faithful,85,The translation maintains the general meaning ...,85.0,good faithfulness and fluency
2,"It inhabits the Atlantic, Indian, and Pacific ...",它居住在大西洋、印度洋、太平洋和地中海。,76.333336,95,Highly faithful and fluent,100,"Flawless translation, maintaining both fluency...",85.0,good faithfulness and fluency
3,The Final 8 began with a victory against Benet...,在 2009 年 4 月 2 日的四分之一决赛中 ， 最后 8 场比赛首先战胜了贝内顿 · ...,56.333332,65,faithful_fluent,98,highly fluent and faithful translation with mi...,NaN,None
4,The verandahs on the west and south elevations...,西面和南面的斜坡上有一排多孔的多孔柱子.,54.666668,20,Misinterpretation of architectural terms,85,"faithfulness is high, but some grammatical err...",85.0,good faithfulness and fluency.
5,"Yet, the first rush of the rebels carried the ...",然而 ， 反叛分子的第一次冲刺占据了 Speen Hill 的阵地。,58.333332,95,Highly accurate but lacks some natural flow,92,Fluent translation with high faithfulness to t...,85.0,good faithfulness and fluency.
6,Both villains wear opulent robes and deck thei...,"这两个恶棍都穿着豪华的长袍, 用铃声装饰他们的交通工具.",55.333332,88,Balanced between faithfulness and fluency,92,"faithful to the original meaning, with minor f...",NaN,None
7,Harding also corresponded with Russian General...,Harding 还与俄罗斯将军 Alexander Kireev 进行了联系。,72.500000,85,Translation is accurate but lacks natural flow,90,faithfulness is high as the main subject and a...,85.0,good faithfulness and fluency
8,It will carry a single cowboy quickly around t...,它将带着一个牛仔迅速在牧场周围做小事.,57.333332,85,faithful_fluent,95,Fluent and highly faithful to the original tex...,85.0,faithfulness and fluency
9,Nicholas Hilliard's miniature of his wife Alic...,尼古拉斯 · 希莉德的妻子爱丽丝的缩影显示她穿着一个开放的部分和一个封闭的地毯。,40.000000,20,"Partial accuracy, incorrect terminology",70,"The translation is mostly faithful, but '地毯' (...",85.0,faithfulness and fluency


In [ ]:
# Export the result data frame as a excel file so thet we don't need to run the file every time when we open the notebook.
Result.to_excel("translation_eval_results.xlsx", index=False)

# 5.Evaluation
In this experiment, we evaluated the performance of the model from two aspects: output stability and consistency with human ratings (correlation and MAE).

In [ ]:
# lode the result we get from three models
# from google.colab import drive
# import pandas as pd

# drive.mount('/content/drive')

# Result = pd.read_excel("/content/drive/MyDrive/translation_eval_results.xlsx")
Result.head()

Mounted at /content/drive


,source_en,translation_zh,human_score,qwen_score,qwen_reason,phi_score,phi_reason,smol_score,smol_reason
0,"Milhaud also used jazz idioms, as in his Suite...","米勒豪德也使用爵士乐成语, 就像他的套房里使用单管、小提琴和钢琴一样.",24.333334,75,"Faithful, fluent",80,The translation maintains the meaning of the o...,0.0,format error
1,He then wrongfooted May by parking just behind...,"然后他把梅的车停在 LDV 后面以阻止它的卸货, 弄错了.",68.500000,80,Fluent and faithful,85,The translation maintains the general meaning ...,85.0,good faithfulness and fluency
2,"It inhabits the Atlantic, Indian, and Pacific ...",它居住在大西洋、印度洋、太平洋和地中海。,76.333336,95,Highly faithful and fluent,100,"Flawless translation, maintaining both fluency...",85.0,good faithfulness and fluency
3,The Final 8 began with a victory against Benet...,在 2009 年 4 月 2 日的四分之一决赛中 ， 最后 8 场比赛首先战胜了贝内顿 · ...,56.333332,65,faithful_fluent,98,highly fluent and faithful translation with mi...,NaN,NaN
4,The verandahs on the west and south elevations...,西面和南面的斜坡上有一排多孔的多孔柱子.,54.666668,20,Misinterpretation of architectural terms,85,"faithfulness is high, but some grammatical err...",85.0,good faithfulness and fluency.


## Count the number of bad outputs for each model

In [ ]:
import pandas as pd
import numpy as np

models = ["qwen", "phi", "smol"]

stats = []

for m in models:
    score_col = f"{m}_score"
    reason_col = f"{m}_reason"

    bad_mask = (
        ((Result[score_col] == 0) & (Result[reason_col] == "format error")) |
        (Result[score_col].isna() & Result[reason_col].isna())
    )

    bad_count = bad_mask.sum()
    total = len(Result)

    stats.append({
        "model": m,
        "total": total,
        "bad_output_count": bad_count,
        "bad_output_rate": bad_count / total,
        "valid_count": total - bad_count
    })

bad_output_stats = pd.DataFrame(stats)
bad_output_stats

,model,total,bad_output_count,bad_output_rate,valid_count
0,qwen,100,0,0.00,100
1,phi,100,7,0.07,93
2,smol,100,40,0.40,60


In terms of output stability, there are significant differences among different models. The Qwen model shows the most stable performance, with no format errors (bad output = 0, error rate 0%) in 100 pieces of data, indicating that it can consistently output structured results as required. In contrast, the Phi model had 7 erroneous outputs (error rate 7%), while the SmolLM model faced the most severe issues, with 40 erroneous outputs (error rate as high as 40%), and only 60 valid data points were available for subsequent analysis. This indicates that as the model size decreases, not only does its evaluation ability decline, but the reliability of its output format also significantly decreases.

In [ ]:
## Correlation & Mean of Difference
for model in ["qwen_score", "phi_score", "smol_score"]:
    reason_col = model.replace("_score", "_reason")

    temp = Result[
        ~(Result[model].isna()) &
        ~((Result[model] == 0) & (Result[reason_col] == "format error"))
    ]

    corr = np.corrcoef(temp["human_score"], temp[model])[0, 1]
    mae = np.mean(np.abs(temp["human_score"] - temp[model]))

    print(model, "Correlation:",corr,"Mean Absolute Error:", mae)

qwen_score Correlation: 0.3841104847245323 Mean Absolute Error: 24.166667098999024
phi_score Correlation: 0.12893167853214627 Mean Absolute Error: 28.03584260838006
smol_score Correlation: 0.04851419058498872 Mean Absolute Error: 25.92222235997518


In terms of consistency with human scoring, we recalculated the indicators after filtering out invalid outputs (NaN and format errors) to make the results more reliable. The correlation of Qwen was 0.384, significantly higher than Phi (0.129) and SmolLM (0.049), indicating that Qwen can to some extent capture the trend of "which translations are better", and possesses certain evaluation capabilities. However, the correlation of Phi was relatively low, showing only a weak correlation; the correlation of SmolLM was close to 0, indicating that its scoring has almost no consistency with human judgments and its sorting ability is very limited.

In terms of error (MAE), the average error of Qwen is approximately 24.17 points, while that of Phi and SmolLM is approximately 28.04 points and 25.92 points respectively. Although the MAE of SmolLM is slightly lower than that of Phi, this result should be interpreted with caution as it is calculated based on only 60 valid data points, which is significantly less than that of Qwen (100 points) and Phi (93 points). Therefore, the reliability of the statistical results is relatively low.

Overall, the performance of the model is not solely determined by the size of the parameters. Although Phi is slightly larger than Qwen, Qwen performs better in Chinese-English understanding, translation evaluation, and structured instruction compliance: it has no format errors and has the highest correlation. In contrast, the stability and consistency of Phi and SmolLM are weaker, especially SmolLM has a large number of format errors. This indicates that translation evaluation not only requires language ability but also a strong ability to follow instructions.

# 6. Analysis of error or failure cases

## Error or Failure cases of Qwen2.5-3B-Instruct
Example:

====== RAW OUTPUT ======

You are a translation evaluator.

English:
But the daemon hordes came again, in far greater numbers and with much more ferocity.

Chinese:
但是雏菊又来了, 数量大得多, 凶猛得多.

Score the translation from 0 to 100 based on:
- faithfulness
- fluency

Return ONLY valid JSON in this format:

{"score": <number>, "reason": "<short phrase>"}
{"score": 0, "reason": "翻译错误，原文是关于Daemon hordes而不是雏菊。"} ```json
{"score": 0, "reason": "翻译错误，原文是关于Daemon hordes而不是雏菊。"}


The overall output of Qwen2.5-3B is better than that of other models. There is no failure in extracting results due to the output format. However, there are some issues with its output results. For example, there are some problems in the output of "reason: short phrase". Six of the reasons are output in Chinese. The reason might be that the model itself comes from a Chinese team. When they trained the model, they also used Chinese for training, and the presence of Chinese in the input prompt led to this result.

## Error or Failure cases of Phi-3.5-mini-instruct

====== RAW OUTPUT ======

You are a translation evaluator.

English:
Ticks pierce the skin of their hosts with specialized mouthparts to suck blood, and they survive exclusively by this obligate method of feeding.

Chinese:
小鸡用专门的口腔刺穿宿主的皮肤吸血 ， 他们完全靠这种强制喂养的方法生存。

Score the translation from 0 to 100 based on:
- faithfulness
- fluency

Return ONLY valid JSON in this format:

{"score": number, "reason": "short phrase"}

For example:

{"score": 90, "reason": "Fluency and faithfulness are high, minor grammatical issues."}

Please provide a detailed evaluation of the translation, considering both the faithfulness to the original text and the fluency of the translated Chinese text.


Answer:

{"score": 92, "reason": "The translation is largely faithful to the original text, capturing the essence of ticks using specialized mouthparts to pierce the skin and survive by blood-feeding. The fluency of the Chinese translation is also good, with natural-sounding phrasing and appropriate use of terminology. Minor adjustments could be made for even greater fluency, such as using "吸血" instead of "吸血" to maintain consistency with the term "feeding" from the original text."}

===============
### Analysis

In this example, although the model appears to generate output in JSON format, the result is actually invalid JSON. The issue occurs in the reason field, where unescaped double quotes are included, such as:

"such as using "吸血" instead of "吸血""
In standard JSON, any double quotes inside a string must be escaped using \". Without proper escaping, the JSON structure becomes malformed. As a result, when the parsing function attempts to read the output using json.loads(), it raises a parsing error.

Due to this failure, the code falls back to the exception handling branch and returns default values (e.g., None or "format error"), which leads to missing scores or incorrect outputs in the final results.

This example highlights a common limitation of LLMs: even when explicitly instructed to produce structured JSON output, they may still violate formatting rules due to natural language generation habits, such as inserting unescaped quotation marks.

## Error or Failure cases of SmolLM2-1.7B-Instruct

====== RAW OUTPUT ======

You are a translation evaluator.

English:
The Final 8 began with a victory against Benetton Treviso in the quarterfinal on 2 April 2009.

Chinese:
在 2009 年 4 月 2 日的四分之一决赛中 ， 最后 8 场比赛首先战胜了贝内顿 · 特雷维索。

Score the translation from 0 to 100 based on:
- faithfulness
- fluency

Return ONLY valid JSON in this format:

{"score": <number>, "reason": "<short phrase>"}

Return an empty object {} if you think the translation is completely uninterpretable.

===============

### Analysis

In this case, the model does not generate an actual answer. Instead, it repeats parts of the prompt, including the JSON template:

{"score": number, "reason": "short phrase"}
As a result, the parser incorrectly extracts this template as if it were a valid output. However, since placeholders like <number> are not valid JSON values, the parsing step fails.

This issue is not simply a formatting error, but a failure of the model to produce any answer at all. Instead of completing the task, the model continues or echoes the prompt. This behavior is more common in smaller models, which may struggle to distinguish between instructions and the expected response, especially when no explicit answer delimiter (e.g., "### Answer:") is provided.

# 7. Summery

Overall, we have largely achieved the project goals. We successfully established a process for evaluating the quality of English-Chinese translations using a small open-source LLM, and compared the model scores with human scores. The results showed that Qwen performed the best, having the highest correlation with human scores (0.384), indicating that it can partially determine the quality of the translation. However, the model is still not reliable enough, as the MAE of Qwen is still approximately 24, suggesting that there is still a significant gap between the scores and human judgments. SmolLM also had 40% of invalid outputs. Overall, small models can provide some reference, but they cannot replace human assessment.

# 8. Limitation and future work

We have encountered some limitations. The main limitation is the model. We originally intended to connect and use the model through the API, but we didn't have the funds to purchase the usage rights. Then we wanted to use some open-source models, such as Qwen/Qwen2-7B-Instruct, and mistralai/Mistral-7B-Instruct-v0.2". However, since the GUP of the Colab notebook cannot run such large models, we finally chose three smaller models and only ran 100 lines of data. Future work could consider using larger models, running more lines of data, and could also consider using a series of models with different parameter sizes. This would allow for a better understanding of the impact of parameter size on the model's output results.